In [ ]:
!export PYTHONPATH=..

from components.utils import read_floats, float2fixed
import pickle
import pandas as pd

In [ ]:
def fixed2float(weights=[], width=16):
    float_weights = []
    for weight in weights:
        sinal = 1 if weight <= ((2**(width-1)) - 1) else -1
        magnitude = weight & 0b0111111111111111
        
        if sinal == -1:
            magnitude = bin(magnitude)[2:].zfill(width)
            magnitude = magnitude.replace("0", "-")
            magnitude = magnitude.replace("1", "0")
            magnitude = magnitude.replace("-", "1")
            magnitude = int(magnitude, 2) + 1

        if width == 16:
            integer_portion = magnitude & 0b0111100000000000
            integer_portion >>= 11
            
            fractional_portion = magnitude & 0b0000011111111111
            float_fraction_portion = 0
            
            for index, char in enumerate(bin(fractional_portion)[2:].zfill(11)):
                float_fraction_portion += 1/(2**(index+1)) if char == "1" else 0
        else:
            integer_portion = magnitude & 0b01111000
            integer_portion >>= 3
            
            fractional_portion = magnitude & 0b00000111
            float_fraction_portion = 0
            
            for index, char in enumerate(bin(fractional_portion)[2:].zfill(3)):
                float_fraction_portion += 1/(2**(index+1)) if char == "1" else 0

        float_value = int(integer_portion) + float_fraction_portion
        float_value *= sinal
        float_weights.append(float_value)
    return float_weights

In [ ]:
weights = read_floats("../binary/weights.pickle")
int_fixed16_weights = float2fixed(weights, integer_portion=4, decimal_portion=11)
int_fixed8_weights = float2fixed(weights, integer_portion=4, decimal_portion=3)
float_fixed16_reconverted = fixed2float(int_fixed16_weights, width=16)
float_fixed8_reconverted = fixed2float(int_fixed8_weights, width=8)

In [ ]:
df = pd.DataFrame({
    "float32": weights,
    "fixed8": float_fixed8_reconverted,
    "fixed16": float_fixed16_reconverted})
df.index = range(df.shape[0])
df["ratio_fixed16"] = df.fixed16 / df.float32
df["error_fixed16"] = df.fixed16 - df.float32
df["ratio_fixed8"] = df.fixed8 / df.float32
df["error_fixed8"] = df.fixed8 - df.float32

pd.set_option('float_format', '{:.6f}'.format)
df.describe()

In [ ]:
df.stack()

In [ ]:
df[["float32", "fixed16", "fixed8"]].plot.box(fontsize=20, figsize=(8, 16), legend=True)

In [ ]:
df[["fixed16", "fixed8", "float32"]].plot(subplots=True, figsize=(16, 16))